In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
import faiss
from scipy.stats import mode
import time

# ======================== KONFIGURASI ========================
# PASTIKAN PATH INI SESUAI DENGAN LOKASI ARTEFAK ANDA DI GOOGLE DRIVE
ARTIFACTS_DIR = "/content/drive/My Drive/colab/bert_faiss_artifacts"
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128
K_NEIGHBORS = 10

# ======================== FUNGSI PREDIKSI ========================
def predict_single_text(text_to_predict, tokenizer, bert_model, faiss_index, train_labels):
    """
    Fungsi untuk mengklasifikasikan satu buah teks.
    """
    print(f"Analyzing text: '{text_to_predict}'...")
    start_time = time.time()

    # 1. Buat embedding dari teks input
    inputs = tokenizer(
        [text_to_predict],
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf"
    )
    outputs = bert_model(inputs)
    embedding = outputs.last_hidden_state[:, 0, :].numpy()
    embedding = embedding.astype('float32')

    # 2. Cari tetangga terdekat di indeks FAISS
    distances, indices = faiss_index.search(embedding, K_NEIGHBORS)

    # 3. Ambil label dari tetangga dan lakukan majority voting
    neighbor_indices = indices[0]
    neighbor_labels = train_labels[neighbor_indices]

    prediction_array, count_array = mode(neighbor_labels)
    predicted_label = int(prediction_array[0])
    vote_count = int(count_array[0])

    confidence = vote_count / K_NEIGHBORS
    end_time = time.time()

    # 4. Tampilkan hasil analisis
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20)
    print(f"Predicted Class: Label {predicted_label}")
    print(f"Confidence: {confidence:.2%} ({vote_count} out of {K_NEIGHBORS} neighbors voted for this class)")
    print(f"Labels of Nearest Neighbors Found: {neighbor_labels}")
    print(f"Inference Time: {end_time - start_time:.4f} seconds")
    print("=" * 57)

    return predicted_label

# ======================== SCRIPT UTAMA ========================
if __name__ == "__main__":
    print("[INFO] Loading necessary artifacts, please wait...")

    try:
        # Muat semua artefak yang diperlukan
        tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
        bert_model = TFBertModel.from_pretrained(MODEL_NAME, from_pt=True)

        faiss_index_path = os.path.join(ARTIFACTS_DIR, "faiss_index.index")
        faiss_index = faiss.read_index(faiss_index_path)

        train_labels_path = os.path.join(ARTIFACTS_DIR, "train_labels.pkl")
        with open(train_labels_path, "rb") as f:
            train_labels = np.array(pickle.load(f))

        print("[SUCCESS] All artifacts loaded. Ready to predict.")
        print("-" * 57)

        # Teks yang ingin dideteksi
        input_text = "fuck you"

        # Panggil fungsi prediksi
        predict_single_text(input_text, tokenizer, bert_model, faiss_index, train_labels)

    except FileNotFoundError as e:
        print(f"\n[ERROR] Artifact file not found: {e}")
        print("Please make sure the path in ARTIFACTS_DIR is correct and your Google Drive is mounted.")
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")